# CatBoost

CatBoost is a gradient boosting algorithm based on decision trees. Due to several differences in its API and data handling compared to previous models, all CatBoost experiments are performed in a separate notebook.

One of the main advantages of CatBoost is its native support for categorical features, eliminating the need for additional preeprocessing techniques such as One-Hot-Encoding. This results in a cleaner and more compact training pipeline while preserving the original categorical information.

This notebook is dedicated exclusively to the CatBoost algorithm. Throughout the following sections, we will establish a baseline model, optimize its hyperparameters using Optuna and evaluate the impact of additional features, including amenities and description embeddings, on the overall prediction performance.

In [1]:
%load_ext autoreload
%autoreload 2

# from time import perf_counter

# NOTEBOOK_START = perf_counter()

import pandas as pd
import numpy as np
import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import catboost
import sklearn

pd.set_option("display.max_columns", 500)

sys.path.append(str(Path().cwd().parent.resolve()))

import preprocessing.features as features

builder = features.FeatureBuilder()

df = pd.read_csv(features.DATASET_PATH)

catboost.__version__

/home/carl/notebooks/airbnb_prices_prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'1.2.10'

In [11]:
ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

TREE_MODELS_RESULTS = ARTIFACTS_DIR / "tree_models_results.csv"

tree_results_df = pd.read_csv(TREE_MODELS_RESULTS, index_col='Model')

tree_results_df.sort_values(by='RMSE', ascending=False)

,MAE,RMSE,R2
Model,,,
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.91,113.39,0.71
"XGBoost (baseline, optimized, zipcode)",49.35,112.69,0.72
"XGBoost (am, em (pca_com=370), optimized (alpha=0.03)",48.82,112.11,0.72
"XGBoost (amenities+embeddings, optimized)",48.68,111.48,0.72
"XGBoost (amenities, embeddings, optimized, zipcode, pca_components=370)",48.33,111.21,0.73


# CatBoost. Baseline Model

In [3]:
from catboost import Pool, CatBoostRegressor
from catboost.utils import get_gpu_device_count

DEVICE = 'GPU' if get_gpu_device_count() > 0 else 'CPU'
DEVICE

'GPU'

In [4]:
df_copy = builder.get_df(df, use_amenities=False, use_embeddings=False)

df_copy.shape

(74111, 27)

In [5]:
X = df_copy.drop(columns='log_price')
y = df_copy['log_price']

X = X.drop(columns=['zipcode'])

X.shape, y.shape

((74111, 25), (74111,))

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    random_state=42,
    test_size=0.2
)

X_train.shape, X_test.shape

((59288, 25), (14823, 25))

In [7]:
%%time

from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

cat_features = X_train.select_dtypes(include=['object', 'string']).columns.tolist()

train_pool = Pool(X_train, label=y_train, cat_features=cat_features)
test_pool = Pool(X_test, label=y_test, cat_features=cat_features)

catboost_baseline = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    task_type=DEVICE,
    random_seed=42,
    verbose=False
)

catboost_baseline.fit(train_pool)

y_pred_test_log = catboost_baseline.predict(test_pool)
y_pred_train_log = catboost_baseline.predict(train_pool)

y_pred_test = np.exp(y_pred_test_log)
y_pred_train = np.exp(y_pred_train_log)

test_MAE = mean_absolute_error(np.exp(y_test), y_pred_test)
train_MAE = mean_absolute_error(np.exp(y_train), y_pred_train)

test_RMSE = root_mean_squared_error(np.exp(y_test), y_pred_test)
train_RMSE = root_mean_squared_error(np.exp(y_train), y_pred_train)

test_r2 = r2_score(y_test, y_pred_test_log)
train_r2 = r2_score(y_train, y_pred_train_log)

print(f"Test MAE: {test_MAE:.2f}$ | Train MAE: {train_MAE:.2f}$")
print(f"Test RMSE: {test_RMSE:.2f}$ | Train RMSE: {train_RMSE:.2f}$")
print(f"Test R2 Score: {test_r2:.2f} | Train R2 Score: {train_r2:.2f}")

Test MAE: 50.28$ | Train MAE: 46.86$
Test RMSE: 113.50$ | Train RMSE: 103.42$
Test R2 Score: 0.71 | Train R2 Score: 0.74
CPU times: user 12.8 s, sys: 1.58 s, total: 14.4 s
Wall time: 9.87 s


In [8]:
catboost_results_df = pd.DataFrame(columns=[
    "MAE", "RMSE", "R2"
])

catboost_results_df.loc['CatBoost baseline'] = [
    round(test_MAE, 2),
    round(test_RMSE, 2),
    round(test_r2, 2)
]

catboost_results_df

,MAE,RMSE,R2
CatBoost baseline,50.28,113.5,0.71


In [12]:
tree_results_df.sort_values(by='RMSE', ascending=False)

,MAE,RMSE,R2
Model,,,
"RandomForest (baseline, optimized)",53.55,121.80,0.67
"XGBoost (baseline, optimized)",49.91,113.39,0.71
"XGBoost (baseline, optimized, zipcode)",49.35,112.69,0.72
"XGBoost (am, em (pca_com=370), optimized (alpha=0.03)",48.82,112.11,0.72
"XGBoost (amenities+embeddings, optimized)",48.68,111.48,0.72
"XGBoost (amenities, embeddings, optimized, zipcode, pca_components=370)",48.33,111.21,0.73


## CatBoost. Baseline, optimized model.

MODELS TO USE TOMORROW, TODO:

1. CatBoost baseline optimized
2. CatBoost full df, pca, zipcode, optimized
3. if prev model overfit, then use objective with alpha